In [ ]:
import os

PROJECT_ROOT = r"C:\Users\vaast\OneDrive\Desktop\UAV_4_Case_Studies\wildfire-prediction-system\src"
config_path = os.path.join(PROJECT_ROOT, "region_config.py")

print(f"🔍 Checking file at: {config_path}")

try:
    with open(config_path, 'r', encoding='utf-8') as file:
        content = file.read()
        print(f"📊 File size: {len(content)} characters")
        
        if "REGIONS" in content:
            print("✅ SUCCESS: The word 'REGIONS' exists in the file text.")
        else:
            print("❌ ERROR: The file exists, but it is either empty or missing the 'REGIONS' dictionary.")
            
        print("\n--- First 150 characters of the file ---")
        print(content[:150])
except Exception as e:
    print(f"Crash reading file: {e}")

In [ ]:
import os
import sys
import importlib.util
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 0️⃣ Direct File Injection (Bypassing Jupyter Cache)
# =========================================================
PROJECT_ROOT = r"C:\Users\vaast\OneDrive\Desktop\UAV_4_Case_Studies\wildfire-prediction-system"
os.chdir(PROJECT_ROOT)

# Define the exact path to the real config file
config_path = os.path.join(PROJECT_ROOT, "src", "region_config.py")

# Force Python to load this specific file, ignoring all caches and other folders
spec = importlib.util.spec_from_file_location("region_config", config_path)
region_config = importlib.util.module_from_spec(spec)
sys.modules["region_config"] = region_config
spec.loader.exec_module(region_config)

REGIONS = region_config.REGIONS

print("📊 LOADING REGIONAL DATA FOR EDA...")
print("=" * 60)
print(f"✅ Loaded {len(REGIONS)} regions: {list(REGIONS.keys())}")

# =========================================================
# 1️⃣ Load All Regional Data
# =========================================================
regional_fires = {}
regional_weather = {}

for region_name in REGIONS.keys():
    try:
        fires = pd.read_csv(
            f"data/case_studies/{region_name}/raw/firms/firms_{region_name}.csv"
        )
        fires['acq_date'] = pd.to_datetime(fires['acq_date'])
        regional_fires[region_name] = fires

        weather = pd.read_csv(
            f"data/case_studies/{region_name}/raw/weather/daily_weather.csv"
        )
        weather['date'] = pd.to_datetime(weather['date'])
        regional_weather[region_name] = weather

        print(f"  ✅ {REGIONS[region_name]['name']}: "
              f"{len(fires)} fire detections | {len(weather)} weather days")

    except FileNotFoundError as e:
        print(f"  ⚠️  Data missing for {region_name}: {e}")

# ... (Keep the rest of your summary statistics and plotting code exactly as it is below here!) ...

# =========================================================
# 2️⃣ Summary Statistics
# =========================================================
print("\n" + "=" * 60)
print("📊 SUMMARY STATISTICS BY REGION")
print("=" * 60)

summary_rows = []
for region_name, fires_df in regional_fires.items():
    if fires_df.empty:
        continue
    row = {
        'Region':              REGIONS[region_name]['name'],
        'Total Detections':    len(fires_df),
        'Avg FRP (MW)':        round(float(fires_df['frp'].mean()), 2),
        'Max FRP (MW)':        round(float(fires_df['frp'].max()), 2),
        'High Conf (>80%)':    int((fires_df['confidence'] > 80).sum()),
        'Date Start':          str(fires_df['acq_date'].min().date()),
        'Date End':            str(fires_df['acq_date'].max().date()),
    }
    summary_rows.append(row)

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))
else:
    print("  No fire data loaded yet.")

# =========================================================
# 3️⃣ Monthly Fire Detection Patterns
# =========================================================
print("\n📈 Generating Monthly Fire Distribution Plots...")

os.makedirs("outputs/comparative_analysis", exist_ok=True)

if regional_fires:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec']

    for idx, (region_name, fires_df) in enumerate(regional_fires.items()):
        ax = axes[idx]

        if fires_df.empty:
            ax.set_title(f"{REGIONS[region_name]['name']}\n(No Data)", fontsize=11)
            continue

        fires_df = fires_df.copy()
        fires_df['month'] = fires_df['acq_date'].dt.month.astype(int)

        # Reindex so all 12 months always appear (even if count is 0)
        monthly = fires_df.groupby('month').size().reindex(range(1, 13), fill_value=0)

        # Colour bars: fire season months in red, others in steelblue
        config = REGIONS[region_name]
        fire_start = int(config['characteristics']['fire_season_start'])
        fire_end   = int(config['characteristics']['fire_season_end'])

        if fire_start <= fire_end:
            fire_months = set(range(fire_start, fire_end + 1))
        else:
            fire_months = set(range(fire_start, 13)) | set(range(1, fire_end + 1))

        bar_colors = [
            'orangered' if m in fire_months else 'steelblue'
            for m in range(1, 13)
        ]

        bars = ax.bar(monthly.index, monthly.values,
                      color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.5)

        ax.set_title(
            f"🔥 {REGIONS[region_name]['name']}\nMonthly Fire Detections",
            fontsize=11, fontweight='bold'
        )
        ax.set_xlabel('Month', fontsize=10)
        ax.set_ylabel('Detections', fontsize=10)
        ax.set_xticks(range(1, 13))
        ax.set_xticklabels(month_labels, fontsize=8)
        ax.grid(alpha=0.3, axis='y')

        # Annotate peak month
        peak_month = int(monthly.idxmax())
        peak_val   = int(monthly.max())
        ax.annotate(
            f'Peak: {month_labels[peak_month - 1]}\n({peak_val})',
            xy=(peak_month, peak_val),
            xytext=(peak_month + 0.5, peak_val * 0.85),
            fontsize=8, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred', lw=1.2)
        )

        # Add legend for fire season colouring
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='orangered', alpha=0.85, label='Fire Season'),
            Patch(facecolor='steelblue', alpha=0.85, label='Off Season')
        ]
        ax.legend(handles=legend_elements, fontsize=8, loc='upper left')

    plt.suptitle('Monthly Fire Detection Patterns — All Regions',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(
        "outputs/comparative_analysis/regional_fire_temporal_patterns.png",
        dpi=300, bbox_inches='tight'
    )
    plt.show()
    print("  ✅ Saved: regional_fire_temporal_patterns.png")

# =========================================================
# 4️⃣ Fire Radiative Power (FRP) Comparison
# =========================================================
print("\n📈 Generating FRP Comparison Plot...")

frp_data = []
for region_name, fires_df in regional_fires.items():
    if not fires_df.empty:
        frp_data.append({
            'Region':   REGIONS[region_name]['name'],
            'Mean FRP': round(float(fires_df['frp'].mean()), 2),
            'Max FRP':  round(float(fires_df['frp'].max()), 2),
            'Std FRP':  round(float(fires_df['frp'].std()), 2),
        })

if frp_data:
    frp_df = pd.DataFrame(frp_data)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart: Mean vs Max FRP
    frp_df.set_index('Region')[['Mean FRP', 'Max FRP']].plot(
        kind='bar', ax=axes[0],
        color=['steelblue', 'orangered'], alpha=0.85, edgecolor='black'
    )
    axes[0].set_title("🔥 Fire Intensity (FRP) by Region",
                      fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Fire Radiative Power (MW)')
    axes[0].set_xlabel('')
    axes[0].grid(alpha=0.3, axis='y')
    axes[0].tick_params(axis='x', rotation=20)

    # Box plot: FRP distribution
    all_frp, all_labels = [], []
    for region_name, fires_df in regional_fires.items():
        if not fires_df.empty:
            valid_frp = fires_df['frp'].dropna().values
            all_frp.append(valid_frp)
            all_labels.append(REGIONS[region_name]['name'])

    if all_frp:
        bp = axes[1].boxplot(all_frp, labels=all_labels, patch_artist=True,
                             notch=False, showfliers=False)
        colors_box = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7']
        for patch, color in zip(bp['boxes'], colors_box):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        axes[1].set_title("📊 FRP Distribution by Region (outliers hidden)",
                          fontsize=12, fontweight='bold')
        axes[1].set_ylabel('Fire Radiative Power (MW)')
        axes[1].tick_params(axis='x', rotation=20)
        axes[1].grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig("outputs/comparative_analysis/frp_comparison.png",
                dpi=300, bbox_inches='tight')
    plt.show()
    print("  ✅ Saved: frp_comparison.png")
    print(frp_df.to_string(index=False))

# =========================================================
# 5️⃣ Weather Overview During Fire Season
# =========================================================
print("\n📈 Generating Weather Comparison Plot...")

weather_summary = []
for region_name, weather_df in regional_weather.items():
    if weather_df.empty:
        continue

    weather_df = weather_df.copy()
    weather_df['date'] = pd.to_datetime(weather_df['date'])
    weather_df['month'] = weather_df['date'].dt.month

    config     = REGIONS[region_name]
    fire_start = int(config['characteristics']['fire_season_start'])
    fire_end   = int(config['characteristics']['fire_season_end'])

    if fire_start <= fire_end:
        fire_months = list(range(fire_start, fire_end + 1))
    else:
        fire_months = list(range(fire_start, 13)) + list(range(1, fire_end + 1))

    fire_season_weather = weather_df[weather_df['month'].isin(fire_months)]

    row = {'Region': REGIONS[region_name]['name']}

    temp_col  = next((c for c in weather_df.columns if 'temperature_2m_mean' in c), None)
    rh_col    = next((c for c in weather_df.columns if 'relative_humidity_2m_mean' in c), None)
    prec_col  = next((c for c in weather_df.columns if 'precipitation_sum' in c), None)
    wind_col  = next((c for c in weather_df.columns if 'wind_speed_10m_max' in c), None)

    row['Temp (°C)']     = round(float(fire_season_weather[temp_col].mean()), 1)  if temp_col  else None
    row['Humidity (%)']  = round(float(fire_season_weather[rh_col].mean()), 1)    if rh_col    else None
    row['Precip (mm)']   = round(float(fire_season_weather[prec_col].mean()), 2)  if prec_col  else None
    row['Wind (km/h)']   = round(float(fire_season_weather[wind_col].mean()), 1)  if wind_col  else None

    weather_summary.append(row)

if weather_summary:
    ws_df = pd.DataFrame(weather_summary).set_index('Region')
    print("\n🌤️  Weather Conditions During Fire Season (Mean Values):")
    print(ws_df.to_string())

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()
    plot_vars = ['Temp (°C)', 'Humidity (%)', 'Precip (mm)', 'Wind (km/h)']
    plot_colors = ['tomato', 'cornflowerblue', 'mediumseagreen', 'mediumpurple']
    icons = ['🌡️', '💧', '🌧️', '🌬️']

    for idx, (var, color, icon) in enumerate(zip(plot_vars, plot_colors, icons)):
        if var not in ws_df.columns:
            continue
        ax = axes[idx]
        ws_df[var].dropna().plot(kind='bar', ax=ax, color=color, alpha=0.85,
                                  edgecolor='black')
        ax.set_title(f"{icon} {var} (Fire Season Avg)", fontsize=11, fontweight='bold')
        ax.set_ylabel(var)
        ax.grid(alpha=0.3, axis='y')
        ax.tick_params(axis='x', rotation=20)

        # Value labels on bars
        for bar in ax.patches:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                f'{bar.get_height():.1f}',
                ha='center', va='bottom', fontsize=9
            )

    plt.suptitle('Weather Conditions During Fire Season — All Regions',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig("outputs/comparative_analysis/regional_weather_comparison.png",
                dpi=300, bbox_inches='tight')
    plt.show()
    print("  ✅ Saved: regional_weather_comparison.png")

# =========================================================
# 6️⃣ Fire Detections by Year
# =========================================================
print("\n📈 Generating Annual Fire Trend Plot...")

if regional_fires:
    fig, ax = plt.subplots(figsize=(12, 5))
    region_colors = {
        'laisong':    '#E63946',
        'jyotikuchi': '#457B9D',
        'corbett':    '#2A9D8F',
        'similipal':  '#E9C46A',
    }

    for region_name, fires_df in regional_fires.items():
        if fires_df.empty:
            continue
        fires_df = fires_df.copy()
        fires_df['year'] = fires_df['acq_date'].dt.year.astype(int)
        annual = fires_df.groupby('year').size()
        ax.plot(
            annual.index, annual.values,
            marker='o', linewidth=2, markersize=6,
            label=REGIONS[region_name]['name'],
            color=region_colors.get(region_name, None)
        )

    ax.set_title("📅 Annual Fire Detection Trends by Region",
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Total Fire Detections')
    ax.legend(fontsize=10, loc='best')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/comparative_analysis/annual_fire_trends.png",
                dpi=300, bbox_inches='tight')
    plt.show()
    print("  ✅ Saved: annual_fire_trends.png")

# =========================================================
# 7️⃣ Confidence Score Distribution
# =========================================================
print("\n📈 Generating Confidence Distribution Plot...")

if regional_fires:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()

    for idx, (region_name, fires_df) in enumerate(regional_fires.items()):
        ax = axes[idx]
        if fires_df.empty:
            continue

        conf = fires_df['confidence'].dropna()
        ax.hist(conf, bins=30, color='darkorange', alpha=0.8, edgecolor='black')
        ax.axvline(x=80, color='red', linestyle='--', linewidth=1.5,
                   label='High Conf Threshold (80%)')
        ax.axvline(x=float(conf.mean()), color='navy', linestyle='-', linewidth=1.5,
                   label=f'Mean = {conf.mean():.1f}%')
        ax.set_title(f"{REGIONS[region_name]['name']}\nFire Detection Confidence",
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Confidence (%)')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3, axis='y')

    plt.suptitle('Fire Detection Confidence Score Distribution — All Regions',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig("outputs/comparative_analysis/confidence_distribution.png",
                dpi=300, bbox_inches='tight')
    plt.show()
    print("  ✅ Saved: confidence_distribution.png")

# =========================================================
# ✅ Final Summary
# =========================================================
print("\n" + "=" * 60)
print("✅ EDA COMPLETE")
print("=" * 60)
print("📁 All charts saved to: outputs/comparative_analysis/")
print("   • regional_fire_temporal_patterns.png")
print("   • frp_comparison.png")
print("   • regional_weather_comparison.png")
print("   • annual_fire_trends.png")
print("   • confidence_distribution.png")
print("\nNext step → Run: notebooks/03_spectral_index_engineering.ipynb")

In [ ]:
# ============================================================
# 02_EDA_TABULAR_BY_REGION_V2.py
# Enhanced Exploratory Data Analysis — Wildfire Prediction System
# Paste each section into a Jupyter cell, or run as a script.
# ============================================================

import os
import sys
import importlib.util
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.0)

# =========================================================
# 0️⃣  PATH SETUP — same pattern as original
# =========================================================
PROJECT_ROOT = r"C:\Users\vaast\OneDrive\Desktop\UAV_4_Case_Studies\wildfire-prediction-system"
os.chdir(PROJECT_ROOT)

config_path = os.path.join(PROJECT_ROOT, "src", "region_config.py")
spec = importlib.util.spec_from_file_location("region_config", config_path)
region_config = importlib.util.module_from_spec(spec)
sys.modules["region_config"] = region_config
spec.loader.exec_module(region_config)
REGIONS = region_config.REGIONS

OUTPUT_DIR = "outputs/comparative_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REGION_COLORS = {
    'laisong':    '#E63946',
    'jyotikuchi': '#457B9D',
    'corbett':    '#2A9D8F',
    'similipal':  '#E9C46A',
}
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

print("📊 ENHANCED EDA — WILDFIRE PREDICTION SYSTEM")
print("=" * 60)
print(f"✅ Loaded {len(REGIONS)} regions: {list(REGIONS.keys())}")

# =========================================================
# 1️⃣  LOAD ALL DATA
# =========================================================
regional_fires   = {}
regional_weather = {}
regional_features = {}

for region_name in REGIONS.keys():
    try:
        fires = pd.read_csv(f"data/case_studies/{region_name}/raw/firms/firms_{region_name}.csv")
        fires['acq_date'] = pd.to_datetime(fires['acq_date'])
        fires['year']  = fires['acq_date'].dt.year.astype(int)
        fires['month'] = fires['acq_date'].dt.month.astype(int)
        fires['doy']   = fires['acq_date'].dt.dayofyear.astype(int)
        regional_fires[region_name] = fires

        weather = pd.read_csv(f"data/case_studies/{region_name}/raw/weather/daily_weather.csv")
        weather['date']  = pd.to_datetime(weather['date'])
        weather['year']  = weather['date'].dt.year.astype(int)
        weather['month'] = weather['date'].dt.month.astype(int)
        regional_weather[region_name] = weather

        feat_path = f"data/case_studies/{region_name}/processed/tabular/features_{region_name}.csv"
        if os.path.exists(feat_path):
            feat = pd.read_csv(feat_path)
            feat['date'] = pd.to_datetime(feat['date'])
            regional_features[region_name] = feat

        print(f"  ✅ {REGIONS[region_name]['name']}: "
              f"{len(fires):,} detections | {len(weather):,} weather days"
              + (f" | {len(regional_features[region_name]):,} feature rows" if region_name in regional_features else ""))
    except FileNotFoundError as e:
        print(f"  ⚠️  Missing: {e}")

# =========================================================
# 2️⃣  SECTION A — MASTER SUMMARY TABLE
# =========================================================
print("\n" + "="*60)
print("📊 MASTER SUMMARY TABLE")
print("="*60)

rows = []
for rn, df in regional_fires.items():
    cfg = REGIONS[rn]
    w   = regional_weather.get(rn, pd.DataFrame())
    row = {
        'Region':           cfg['name'],
        'State':            cfg['state'],
        'Area (km²)':       cfg['area_sq_km'],
        'Total Detections': len(df),
        'Years Covered':    f"{df['year'].min()}–{df['year'].max()}",
        'Avg FRP (MW)':     round(df['frp'].mean(), 2),
        'Max FRP (MW)':     round(df['frp'].max(), 2),
        'Mean Conf (%)':    round(df['confidence'].mean(), 1),
        'High Conf >80%':   int((df['confidence'] > 80).sum()),
        'Peak Month':       MONTH_LABELS[int(df.groupby('month').size().idxmax()) - 1],
        'Fire Season':      f"M{cfg['characteristics']['fire_season_start']}–M{cfg['characteristics']['fire_season_end']}",
    }
    if not w.empty:
        temp_col = next((c for c in w.columns if 'temperature_2m_mean' in c), None)
        if temp_col:
            row['Avg Temp (°C)'] = round(w[temp_col].mean(), 1)
    rows.append(row)

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# =========================================================
# 3️⃣  SECTION B — FIRE DETECTION DENSITY (Detections/km²)
# =========================================================
print("\n📈 Plot B: Fire Detection Density...")

density_data = []
for rn, df in regional_fires.items():
    area = REGIONS[rn]['area_sq_km']
    density_data.append({
        'Region': REGIONS[rn]['name'],
        'Total': len(df),
        'Per km²': round(len(df) / area, 4),
        'Per km² per Year': round(len(df) / area / max(df['year'].nunique(), 1), 4),
    })
density_df = pd.DataFrame(density_data)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [REGION_COLORS[rn] for rn in REGIONS.keys()]

axes[0].bar(density_df['Region'], density_df['Total'], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title("Total Fire Detections", fontweight='bold')
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[0].patches, density_df['Total']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, f'{val:,}', ha='center', fontsize=8)

axes[1].bar(density_df['Region'], density_df['Per km²'], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title("Detections per km²", fontweight='bold')
axes[1].set_ylabel("Detections / km²")
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[1].patches, density_df['Per km²']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, f'{val:.4f}', ha='center', fontsize=8)

axes[2].bar(density_df['Region'], density_df['Per km² per Year'], color=colors, edgecolor='black', alpha=0.85)
axes[2].set_title("Detections per km² per Year", fontweight='bold')
axes[2].set_ylabel("Detections / km² / yr")
axes[2].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[2].patches, density_df['Per km² per Year']):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, f'{val:.4f}', ha='center', fontsize=8)

plt.suptitle("Fire Detection Density Normalised by Area & Time", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/B_fire_density_normalised.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: B_fire_density_normalised.png")

# =========================================================
# 4️⃣  SECTION C — HEATMAP: Month × Year Fire Counts
# =========================================================
print("\n📈 Plot C: Month × Year Heatmaps...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for idx, (rn, df) in enumerate(regional_fires.items()):
    pivot = df.groupby(['year','month']).size().unstack(fill_value=0)
    pivot.columns = [MONTH_LABELS[m-1] for m in pivot.columns]
    
    sns.heatmap(pivot, ax=axes[idx], cmap='YlOrRd', linewidths=0.4,
                linecolor='white', annot=True, fmt='d', annot_kws={'size':7},
                cbar_kws={'label':'Detections'})
    axes[idx].set_title(f"{REGIONS[rn]['name']}\nYearly × Monthly Fire Detections",
                        fontweight='bold', fontsize=11)
    axes[idx].set_xlabel("Month")
    axes[idx].set_ylabel("Year")
    axes[idx].tick_params(axis='x', rotation=30)

plt.suptitle("Fire Detections: Year × Month Heatmaps — All Regions",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/C_month_year_heatmaps.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: C_month_year_heatmaps.png")

# =========================================================
# 5️⃣  SECTION D — DAY-OF-YEAR DENSITY (KDE) CURVES
# =========================================================
print("\n📈 Plot D: Day-of-Year KDE curves...")

fig, ax = plt.subplots(figsize=(14, 5))

for rn, df in regional_fires.items():
    kde = stats.gaussian_kde(df['doy'], bw_method=0.1)
    x = np.linspace(1, 365, 500)
    y = kde(x)
    ax.fill_between(x, y, alpha=0.18, color=REGION_COLORS[rn])
    ax.plot(x, y, color=REGION_COLORS[rn], lw=2, label=REGIONS[rn]['name'])

# Add month boundary lines
for m in range(1, 13):
    doy = pd.Timestamp(f"2021-{m:02d}-01").dayofyear
    ax.axvline(doy, color='grey', lw=0.6, linestyle='--', alpha=0.5)
    ax.text(doy+1, ax.get_ylim()[1]*0.95, MONTH_LABELS[m-1], fontsize=7, color='grey', va='top')

ax.set_xlabel("Day of Year")
ax.set_ylabel("Probability Density")
ax.set_title("Fire Detection Density Across the Calendar Year (KDE)", fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/D_doy_kde_curves.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: D_doy_kde_curves.png")

# =========================================================
# 6️⃣  SECTION E — FRP VIOLIN PLOTS + STATISTICAL TESTS
# =========================================================
print("\n📈 Plot E: FRP Violin Plots + Stats...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

frp_list   = []
label_list = []
for rn, df in regional_fires.items():
    valid = df['frp'].dropna()
    frp_list.append(valid.values)
    label_list.append(REGIONS[rn]['name'])

# Violin
parts = axes[0].violinplot(frp_list, positions=range(1, len(frp_list)+1),
                            showmedians=True, showextrema=True)
colors_v = list(REGION_COLORS.values())
for pc, col in zip(parts['bodies'], colors_v):
    pc.set_facecolor(col)
    pc.set_alpha(0.7)
axes[0].set_xticks(range(1, len(label_list)+1))
axes[0].set_xticklabels(label_list, rotation=15, ha='right')
axes[0].set_ylabel("FRP (MW)")
axes[0].set_title("FRP Distribution — Violin Plot", fontweight='bold')
axes[0].set_ylim(0, 50)
axes[0].text(0.01, 0.98, "Y-axis capped at 50 MW for clarity",
             transform=axes[0].transAxes, fontsize=7, va='top', color='grey')

# Log-scale boxplot
bp = axes[1].boxplot(frp_list, labels=label_list, patch_artist=True, showfliers=True,
                      flierprops=dict(marker='.', markersize=2, alpha=0.3))
for patch, col in zip(bp['boxes'], colors_v):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
axes[1].set_yscale('log')
axes[1].set_ylabel("FRP (MW) — log scale")
axes[1].set_title("FRP Distribution — Log Scale Boxplot (all outliers shown)", fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle("Fire Radiative Power Distribution — All Regions", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/E_frp_violin_logbox.png", dpi=300, bbox_inches='tight')
plt.show()

# Print percentile table
print("\n  📊 FRP Percentile Table (MW):")
pct_rows = []
for rn, df in regional_fires.items():
    v = df['frp'].dropna()
    pct_rows.append({
        'Region':  REGIONS[rn]['name'],
        'P25':     round(v.quantile(0.25), 2),
        'Median':  round(v.median(), 2),
        'Mean':    round(v.mean(), 2),
        'P75':     round(v.quantile(0.75), 2),
        'P90':     round(v.quantile(0.90), 2),
        'P99':     round(v.quantile(0.99), 2),
        'Max':     round(v.max(), 2),
        'Skew':    round(float(stats.skew(v)), 2),
    })
print(pd.DataFrame(pct_rows).to_string(index=False))
print("  ✅ Saved: E_frp_violin_logbox.png")

# Kruskal-Wallis test (non-parametric ANOVA across regions)
kw_stat, kw_p = stats.kruskal(*frp_list)
print(f"\n  📐 Kruskal-Wallis test (FRP differs across regions?):")
print(f"     H = {kw_stat:.2f}, p = {kw_p:.2e}  → {'Significant ✅' if kw_p < 0.05 else 'Not significant'}")

# =========================================================
# 7️⃣  SECTION F — CONFIDENCE SCORE DEEP DIVE
# =========================================================
print("\n📈 Plot F: Confidence Score Deep Dive...")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Per-region CDF of confidence
for idx, (rn, df) in enumerate(regional_fires.items()):
    ax = axes[0, idx] if idx < 3 else axes[1, idx-3]
    conf = df['confidence'].dropna().sort_values()
    cdf  = np.arange(1, len(conf)+1) / len(conf)
    ax.plot(conf, cdf, color=REGION_COLORS[rn], lw=2)
    ax.axvline(80, color='red', linestyle='--', lw=1.5, label='80% threshold')
    ax.axvline(float(conf.mean()), color='navy', linestyle='-', lw=1.5,
               label=f"Mean={conf.mean():.1f}%")
    pct_above = (conf > 80).mean() * 100
    ax.text(0.05, 0.92, f"{pct_above:.1f}% above 80%", transform=ax.transAxes,
            fontsize=9, color='darkred')
    ax.set_title(f"{REGIONS[rn]['name']}\nConfidence CDF", fontweight='bold', fontsize=10)
    ax.set_xlabel("Confidence (%)")
    ax.set_ylabel("Cumulative Proportion")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# Row 2, last 2 panels: Overlay KDE + confidence by FRP bucket
ax_kde  = axes[1, 0]
ax_frp_conf = axes[1, 1]
ax_scatter  = axes[1, 2]

for rn, df in regional_fires.items():
    conf = df['confidence'].dropna()
    kde = stats.gaussian_kde(conf, bw_method=0.15)
    x = np.linspace(conf.min(), conf.max(), 300)
    ax_kde.plot(x, kde(x), color=REGION_COLORS[rn], lw=2, label=REGIONS[rn]['name'])
    ax_kde.fill_between(x, kde(x), alpha=0.1, color=REGION_COLORS[rn])
ax_kde.axvline(80, color='red', linestyle='--', lw=1.5)
ax_kde.set_title("Confidence KDE — All Regions Overlaid", fontweight='bold')
ax_kde.set_xlabel("Confidence (%)")
ax_kde.set_ylabel("Density")
ax_kde.legend(fontsize=8)

# FRP vs Confidence scatter
all_rn_list, all_frp_list, all_conf_list = [], [], []
for rn, df in regional_fires.items():
    sub = df[['frp','confidence']].dropna()
    all_rn_list.extend([REGIONS[rn]['name']]*len(sub))
    all_frp_list.extend(sub['frp'].tolist())
    all_conf_list.extend(sub['confidence'].tolist())
    ax_frp_conf.scatter(sub['confidence'], np.log1p(sub['frp']),
                        color=REGION_COLORS[rn], alpha=0.15, s=5, label=REGIONS[rn]['name'])

ax_frp_conf.set_xlabel("Confidence (%)")
ax_frp_conf.set_ylabel("log(1+FRP)")
ax_frp_conf.set_title("Confidence vs FRP (log)", fontweight='bold')
ax_frp_conf.legend(fontsize=7, markerscale=3)

r, p = pearsonr(all_conf_list, np.log1p(all_frp_list))
ax_frp_conf.text(0.04, 0.94, f"r = {r:.2f}, p = {p:.2e}", transform=ax_frp_conf.transAxes, fontsize=8)

# Confidence by month (all regions)
monthly_conf_by_region = {}
for rn, df in regional_fires.items():
    monthly_conf_by_region[REGIONS[rn]['name']] = df.groupby('month')['confidence'].mean()

for rn_name, series in monthly_conf_by_region.items():
    rn_key = [k for k,v in REGIONS.items() if v['name'] == rn_name][0]
    ax_scatter.plot(series.index, series.values, marker='o', color=REGION_COLORS[rn_key],
                    label=rn_name, lw=2)
ax_scatter.set_xticks(range(1,13))
ax_scatter.set_xticklabels(MONTH_LABELS, fontsize=8)
ax_scatter.set_xlabel("Month")
ax_scatter.set_ylabel("Mean Confidence (%)")
ax_scatter.set_title("Mean Confidence by Month", fontweight='bold')
ax_scatter.legend(fontsize=7)
ax_scatter.grid(alpha=0.3)

plt.suptitle("Confidence Score Deep Dive — All Regions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/F_confidence_deep_dive.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: F_confidence_deep_dive.png")

# =========================================================
# 8️⃣  SECTION G — WEATHER CORRELATION MATRIX (per region)
# =========================================================
print("\n📈 Plot G: Weather Correlation Matrices...")

weather_feature_keywords = [
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min',
    'relative_humidity_2m_mean', 'precipitation_sum',
    'wind_speed_10m_max', 'wind_speed_10m_mean',
    'soil_moisture_0_1cm_mean', 'et0_fao_evapotranspiration_sum'
]
short_names = {
    'temperature_2m_mean':             'Temp Mean',
    'temperature_2m_max':              'Temp Max',
    'temperature_2m_min':              'Temp Min',
    'relative_humidity_2m_mean':       'RH Mean',
    'precipitation_sum':               'Precip',
    'wind_speed_10m_max':              'Wind Max',
    'wind_speed_10m_mean':             'Wind Mean',
    'soil_moisture_0_1cm_mean':        'Soil Moist.',
    'et0_fao_evapotranspiration_sum':  'ET0',
}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (rn, wdf) in enumerate(regional_weather.items()):
    cols = [c for c in wdf.columns for kw in weather_feature_keywords if kw in c]
    cols = list(dict.fromkeys(cols))  # deduplicate
    if len(cols) < 3:
        continue
    sub = wdf[cols].dropna()
    rename_map = {c: next((short_names[kw] for kw in weather_feature_keywords if kw in c), c)
                  for c in cols}
    sub = sub.rename(columns=rename_map)
    corr = sub.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=axes[idx], mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, linewidths=0.5, annot_kws={'size':7},
                cbar_kws={'shrink':0.8})
    axes[idx].set_title(f"{REGIONS[rn]['name']}\nWeather Feature Correlation", fontweight='bold')
    axes[idx].tick_params(axis='x', rotation=40, labelsize=8)
    axes[idx].tick_params(axis='y', labelsize=8)

plt.suptitle("Weather Variable Correlation Matrices — All Regions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/G_weather_correlation_matrices.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: G_weather_correlation_matrices.png")

# =========================================================
# 9️⃣  SECTION H — FIRE vs WEATHER: MERGED TIME-SERIES
# =========================================================
print("\n📈 Plot H: Fire Counts vs Weather Variables (time-series)...")

fig, big_axes = plt.subplots(4, 1, figsize=(18, 22))

for ax_main, (rn, fires_df) in zip(big_axes, regional_fires.items()):
    wdf   = regional_weather.get(rn, pd.DataFrame())
    if wdf.empty:
        continue

    # Monthly aggregation
    monthly_fires = fires_df.groupby(['year','month']).size().reset_index(name='count')
    monthly_fires['date'] = pd.to_datetime(
        monthly_fires.apply(lambda r: f"{int(r['year'])}-{int(r['month']):02d}-01", axis=1))
    monthly_fires = monthly_fires.sort_values('date')

    temp_col = next((c for c in wdf.columns if 'temperature_2m_mean' in c), None)
    rh_col   = next((c for c in wdf.columns if 'relative_humidity_2m_mean' in c), None)
    prec_col = next((c for c in wdf.columns if 'precipitation_sum' in c), None)

    wdf_monthly = wdf.groupby(['year','month']).agg(
        temp=(temp_col, 'mean') if temp_col else ('date','count'),
        rh=(rh_col, 'mean') if rh_col else ('date','count'),
        prec=(prec_col, 'sum') if prec_col else ('date','count'),
    ).reset_index()
    wdf_monthly['date'] = pd.to_datetime(
        wdf_monthly.apply(lambda r: f"{int(r['year'])}-{int(r['month']):02d}-01", axis=1))

    merged = pd.merge(monthly_fires, wdf_monthly, on='date', how='inner')

    # Twin axes
    color_fire = REGION_COLORS[rn]
    ax2 = ax_main.twinx()
    ax3 = ax_main.twinx()
    ax3.spines['right'].set_position(('outward', 60))

    ax_main.bar(merged['date'], merged['count'], width=25, alpha=0.55,
                color=color_fire, label='Fire Detections')
    if temp_col:
        ax2.plot(merged['date'], merged['temp'], color='tomato', lw=1.5,
                 linestyle='--', label='Temp (°C)')
    if rh_col:
        ax3.plot(merged['date'], merged['rh'], color='steelblue', lw=1.5,
                 linestyle=':', label='Humidity (%)')

    ax_main.set_ylabel("Fire Detections", color=color_fire)
    ax2.set_ylabel("Temp (°C)", color='tomato')
    ax3.set_ylabel("Humidity (%)", color='steelblue')
    ax_main.set_title(f"{REGIONS[rn]['name']} — Monthly Fire Count vs Temperature & Humidity",
                      fontweight='bold', fontsize=11)

    lines1, labels1 = ax_main.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    lines3, labels3 = ax3.get_legend_handles_labels()
    ax_main.legend(lines1+lines2+lines3, labels1+labels2+labels3, fontsize=8, loc='upper right')
    ax_main.grid(alpha=0.2)

plt.suptitle("Monthly Fire Counts vs Weather Variables — 2018–2026", fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/H_fire_vs_weather_timeseries.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: H_fire_vs_weather_timeseries.png")

# =========================================================
# 🔟  SECTION I — FIRE SEASON vs OFF-SEASON STATS
# =========================================================
print("\n📈 Plot I: Fire Season vs Off-Season Weather Box Comparison...")

def get_fire_months(rn):
    cfg = REGIONS[rn]['characteristics']
    s, e = int(cfg['fire_season_start']), int(cfg['fire_season_end'])
    return set(range(s,13)) | set(range(1,e+1)) if s > e else set(range(s,e+1))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

weather_cols_to_compare = {
    'temperature': 'temperature_2m_mean',
    'humidity':    'relative_humidity_2m_mean',
}
row_labels = list(weather_cols_to_compare.keys())

for col_idx, (rn, wdf) in enumerate(regional_weather.items()):
    fire_months = get_fire_months(rn)
    wdf = wdf.copy()
    wdf['season'] = wdf['month'].apply(lambda m: 'Fire Season' if m in fire_months else 'Off Season')

    for row_idx, (var_label, col_kw) in enumerate(weather_cols_to_compare.items()):
        col_name = next((c for c in wdf.columns if col_kw in c), None)
        ax = axes[row_idx, col_idx]
        if col_name is None:
            ax.set_visible(False)
            continue

        groups = [wdf[wdf['season']=='Fire Season'][col_name].dropna().values,
                  wdf[wdf['season']=='Off Season'][col_name].dropna().values]
        bp = ax.boxplot(groups, labels=['Fire\nSeason','Off\nSeason'],
                        patch_artist=True, showfliers=False)
        bp['boxes'][0].set_facecolor('orangered')
        bp['boxes'][0].set_alpha(0.7)
        bp['boxes'][1].set_facecolor('steelblue')
        bp['boxes'][1].set_alpha(0.7)

        # T-test
        t_stat, t_p = stats.ttest_ind(groups[0], groups[1])
        ax.set_title(f"{REGIONS[rn]['name'][:12]}\n{var_label.title()} (p={t_p:.2e})",
                     fontsize=9, fontweight='bold')
        ax.set_ylabel(var_label.replace('_',' ').title(), fontsize=8)
        sig_text = '***' if t_p < 0.001 else ('**' if t_p < 0.01 else ('*' if t_p < 0.05 else 'ns'))
        ax.text(1.5, ax.get_ylim()[1]*0.98, sig_text, ha='center', fontsize=12, color='darkred')
        ax.grid(alpha=0.3, axis='y')

plt.suptitle("Fire Season vs Off-Season Weather Comparison (T-test significance shown)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/I_fire_vs_offseason_weather.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: I_fire_vs_offseason_weather.png")

# =========================================================
# 1️⃣1️⃣  SECTION J — ROLLING 30-DAY FIRE FREQUENCY (CLIMATOLOGY)
# =========================================================
print("\n📈 Plot J: 30-day Rolling Fire Climatology (DOY averaged across years)...")

fig, ax = plt.subplots(figsize=(14, 6))

for rn, df in regional_fires.items():
    # Build a day-of-year count averaged over all years
    doy_counts = df.groupby(['year','doy']).size().reset_index(name='count')
    avg_by_doy = doy_counts.groupby('doy')['count'].mean().reindex(range(1,366), fill_value=0)
    rolling    = avg_by_doy.rolling(30, center=True, min_periods=1).mean()
    ax.plot(rolling.index, rolling.values, color=REGION_COLORS[rn], lw=2.5,
            label=REGIONS[rn]['name'])
    ax.fill_between(rolling.index, rolling.values, alpha=0.1, color=REGION_COLORS[rn])

# Month separators
for m in range(1, 13):
    doy_m = pd.Timestamp(f"2021-{m:02d}-01").dayofyear
    ax.axvline(doy_m, color='grey', lw=0.5, linestyle='--', alpha=0.5)
    ax.text(doy_m+1, ax.get_ylim()[1]*0.97, MONTH_LABELS[m-1], fontsize=8, color='grey', va='top')

ax.set_xlabel("Day of Year")
ax.set_ylabel("Avg Daily Detections (30-day rolling mean)")
ax.set_title("Multi-Year Fire Climatology — 30-Day Rolling Average", fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/J_fire_climatology_rolling.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: J_fire_climatology_rolling.png")

# =========================================================
# 1️⃣2️⃣  SECTION K — FIRE-WEATHER SCATTER MATRIX (Seaborn Pairplot-style)
# =========================================================
print("\n📈 Plot K: Fire Frequency vs Key Weather Variables (scatter + regression)...")

fig, big_ax = plt.subplots(4, 3, figsize=(18, 22))

weather_scatter_vars = {
    'Temp (°C)':   'temperature_2m_mean',
    'Humidity (%)': 'relative_humidity_2m_mean',
    'Precip (mm)': 'precipitation_sum',
}

for row_idx, (rn, fires_df) in enumerate(regional_fires.items()):
    wdf = regional_weather.get(rn, pd.DataFrame())
    if wdf.empty:
        continue
    monthly_fires = fires_df.groupby(['year','month']).size().reset_index(name='fire_count')

    for col_idx, (var_label, col_kw) in enumerate(weather_scatter_vars.items()):
        ax = big_ax[row_idx, col_idx]
        col_name = next((c for c in wdf.columns if col_kw in c), None)
        if col_name is None:
            ax.set_visible(False)
            continue

        w_monthly = wdf.groupby(['year','month'])[[col_name]].mean().reset_index()
        merged = pd.merge(monthly_fires, w_monthly, on=['year','month'])

        x, y = merged[col_name].values, merged['fire_count'].values
        ax.scatter(x, y, color=REGION_COLORS[rn], alpha=0.6, s=20, edgecolors='none')

        # Regression line
        if len(x) > 5:
            slope, intercept, r, p, _ = stats.linregress(x, y)
            x_line = np.linspace(x.min(), x.max(), 100)
            ax.plot(x_line, slope*x_line+intercept, 'k--', lw=1.5)
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            ax.text(0.05, 0.92, f"r={r:.2f} {sig}", transform=ax.transAxes, fontsize=9,
                    color='darkred', fontweight='bold')

        ax.set_xlabel(var_label, fontsize=9)
        ax.set_ylabel("Monthly Fire Detections", fontsize=9)
        ax.set_title(f"{REGIONS[rn]['name'][:14]}\nvs {var_label}", fontsize=9, fontweight='bold')
        ax.grid(alpha=0.3)

plt.suptitle("Monthly Fire Detections vs Weather Variables — Scatter + Regression", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/K_fire_weather_scatter_matrix.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: K_fire_weather_scatter_matrix.png")

# =========================================================
# 1️⃣3️⃣  SECTION L — YEAR-ON-YEAR % CHANGE
# =========================================================
print("\n📈 Plot L: Year-on-Year % Change in Fire Detections...")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (rn, df) in enumerate(regional_fires.items()):
    annual = df.groupby('year').size()
    pct_change = annual.pct_change() * 100

    ax = axes[idx]
    colors_bar = ['forestgreen' if v <= 0 else 'firebrick' for v in pct_change.fillna(0)]
    ax.bar(pct_change.index, pct_change.values, color=colors_bar, edgecolor='black',
           alpha=0.8, width=0.7)
    ax.axhline(0, color='black', lw=1.0)
    ax.set_title(f"{REGIONS[rn]['name']}\nYear-on-Year % Change in Fire Detections",
                 fontweight='bold', fontsize=10)
    ax.set_xlabel("Year")
    ax.set_ylabel("% Change")
    ax.grid(alpha=0.3, axis='y')

    for x_val, y_val in zip(pct_change.index, pct_change.values):
        if not np.isnan(y_val):
            ax.text(x_val, y_val + (3 if y_val >= 0 else -8), f"{y_val:+.0f}%",
                    ha='center', fontsize=8)

plt.suptitle("Year-on-Year % Change in Fire Detections (Green = Decrease, Red = Increase)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/L_yoy_pct_change.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: L_yoy_pct_change.png")

# =========================================================
# 1️⃣4️⃣  SECTION M — ENGINEERED FEATURES EDA (if available)
# =========================================================
if regional_features:
    print("\n📈 Plot M: Engineered Feature Distributions (VPD, NDVI, Fire Labels)...")

    fig, axes = plt.subplots(3, 4, figsize=(20, 14))

    feat_to_plot = ['vpd', 'ndvi', 'nbr', 'fuel_dryness',
                    'temperature_2m_mean', 'relative_humidity_2m_mean',
                    'precipitation_sum', 'wind_speed_10m_max',
                    'ndvi_anomaly', 'nbr_deficit', 'soil_moisture_0_1cm_mean', 'fire_season']

    for col_idx, feat in enumerate(feat_to_plot):
        ax = axes[col_idx // 4, col_idx % 4]
        plotted = False
        for rn, fdf in regional_features.items():
            if feat in fdf.columns:
                vals = fdf[feat].dropna()
                if vals.nunique() > 2:
                    kde = stats.gaussian_kde(vals, bw_method=0.15)
                    x = np.linspace(vals.min(), vals.max(), 200)
                    ax.fill_between(x, kde(x), alpha=0.15, color=REGION_COLORS[rn])
                    ax.plot(x, kde(x), color=REGION_COLORS[rn], lw=2, label=REGIONS[rn]['name'])
                else:
                    counts = fdf[feat].value_counts(normalize=True)
                    ax.bar([str(k) for k in counts.index], counts.values,
                           color=REGION_COLORS[rn], alpha=0.5, width=0.3, label=REGIONS[rn]['name'])
                plotted = True
        if plotted:
            ax.set_title(feat.replace('_', ' ').title(), fontweight='bold', fontsize=9)
            ax.legend(fontsize=6)
            ax.grid(alpha=0.3)
        else:
            ax.set_visible(False)

    plt.suptitle("Engineered Feature Distributions — All Regions", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/M_engineered_features_dist.png", dpi=300, bbox_inches='tight')
    plt.show()
    print("  ✅ Saved: M_engineered_features_dist.png")

    # --- Fire vs No-Fire feature separation ---
    print("\n📈 Plot M2: Fire vs No-Fire Feature Separation...")

    key_features = ['vpd', 'temperature_2m_mean', 'relative_humidity_2m_mean',
                    'wind_speed_10m_max', 'ndvi', 'soil_moisture_0_1cm_mean']

    fig, axes = plt.subplots(len(regional_features), len(key_features),
                              figsize=(22, 4*len(regional_features)))

    for row_idx, (rn, fdf) in enumerate(regional_features.items()):
        if 'fire' not in fdf.columns:
            continue
        for col_idx, feat in enumerate(key_features):
            ax = axes[row_idx, col_idx] if len(regional_features) > 1 else axes[col_idx]
            if feat not in fdf.columns:
                ax.set_visible(False)
                continue
            fire_vals    = fdf[fdf['fire']==1][feat].dropna()
            no_fire_vals = fdf[fdf['fire']==0][feat].dropna()

            for vals, label, color in [(fire_vals,'Fire','orangered'),(no_fire_vals,'No Fire','steelblue')]:
                if len(vals) > 5:
                    kde = stats.gaussian_kde(vals, bw_method=0.2)
                    x = np.linspace(vals.min(), vals.max(), 200)
                    ax.fill_between(x, kde(x), alpha=0.25, color=color)
                    ax.plot(x, kde(x), color=color, lw=2, label=label)

            # Mann-Whitney U test
            if len(fire_vals) > 5 and len(no_fire_vals) > 5:
                u_stat, u_p = stats.mannwhitneyu(fire_vals, no_fire_vals, alternative='two-sided')
                sig = '***' if u_p < 0.001 else ('**' if u_p < 0.01 else ('*' if u_p < 0.05 else 'ns'))
                ax.text(0.05, 0.92, sig, transform=ax.transAxes, fontsize=11, color='darkred', fontweight='bold')

            ax.set_title(f"{REGIONS[rn]['name'][:12]}\n{feat.replace('_',' ')}", fontsize=8, fontweight='bold')
            ax.legend(fontsize=7)
            ax.grid(alpha=0.3)

    plt.suptitle("Fire vs No-Fire Feature Distributions (*** = highly significant separation)",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/M2_fire_nofire_separation.png", dpi=300, bbox_inches='tight')
    plt.show()
    print("  ✅ Saved: M2_fire_nofire_separation.png")

    # --- Class Imbalance Summary ---
    print("\n  📊 Class Imbalance Summary:")
    imb_rows = []
    for rn, fdf in regional_features.items():
        if 'fire' not in fdf.columns:
            continue
        vc = fdf['fire'].value_counts()
        total = len(fdf)
        fire_n    = int(vc.get(1, 0))
        nofire_n  = int(vc.get(0, 0))
        ratio = round(nofire_n / max(fire_n, 1), 1)
        imb_rows.append({
            'Region':       REGIONS[rn]['name'],
            'Fire Days':    fire_n,
            'No-Fire Days': nofire_n,
            'Total':        total,
            'Fire %':       round(fire_n/total*100, 1),
            'Imbalance Ratio': f"1:{ratio}",
        })
    print(pd.DataFrame(imb_rows).to_string(index=False))

# =========================================================
# 1️⃣5️⃣  SECTION N — SPATIAL COORDINATE SCATTER (fire locations)
# =========================================================
print("\n📈 Plot N: Fire Detection Spatial Scatter...")

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, (rn, df) in enumerate(regional_fires.items()):
    ax = axes[idx]
    cfg = REGIONS[rn]
    bounds = cfg['bounds']

    if 'latitude' not in df.columns or 'longitude' not in df.columns:
        ax.set_visible(False)
        continue

    sc = ax.scatter(df['longitude'], df['latitude'],
                    c=df['frp'], cmap='hot_r', s=5, alpha=0.4,
                    vmin=0, vmax=df['frp'].quantile(0.95))
    plt.colorbar(sc, ax=ax, label='FRP (MW)')

    # Bounding box
    rect = plt.Rectangle((bounds['lon_min'], bounds['lat_min']),
                          bounds['lon_max']-bounds['lon_min'],
                          bounds['lat_max']-bounds['lat_min'],
                          linewidth=2, edgecolor='blue', facecolor='none')
    ax.add_patch(rect)
    ax.plot(cfg['center']['lon'], cfg['center']['lat'], 'b^', markersize=8)

    ax.set_title(f"{cfg['name']}\nFire Detection Locations (coloured by FRP)",
                 fontweight='bold', fontsize=10)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xlim(bounds['lon_min']-0.05, bounds['lon_max']+0.05)
    ax.set_ylim(bounds['lat_min']-0.05, bounds['lat_max']+0.05)
    ax.grid(alpha=0.3)

plt.suptitle("Spatial Distribution of Fire Detections (FRP intensity coloured)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/N_spatial_fire_scatter.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: N_spatial_fire_scatter.png")

# =========================================================
# 1️⃣6️⃣  SECTION O — CROSS-REGION FIRE SYNCHRONY
# =========================================================
print("\n📈 Plot O: Cross-Region Fire Synchrony (Correlation of Annual Trends)...")

annual_series = {}
for rn, df in regional_fires.items():
    annual = df.groupby('year').size().rename(REGIONS[rn]['name'])
    annual_series[REGIONS[rn]['name']] = annual

annual_df = pd.DataFrame(annual_series).dropna()
corr_annual = annual_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(corr_annual, ax=axes[0], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1, cbar_kws={'shrink':0.8})
axes[0].set_title("Annual Fire Count Correlation\nAcross Regions", fontweight='bold')

# Monthly synchrony
monthly_series = {}
for rn, df in regional_fires.items():
    m_counts = df.groupby(['year','month']).size()
    monthly_series[REGIONS[rn]['name']] = m_counts

monthly_df = pd.DataFrame(monthly_series).dropna()
corr_monthly = monthly_df.corr()
sns.heatmap(corr_monthly, ax=axes[1], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1, cbar_kws={'shrink':0.8})
axes[1].set_title("Monthly Fire Count Correlation\nAcross Regions", fontweight='bold')

plt.suptitle("Cross-Region Fire Synchrony — Are Regions Burning Together?",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/O_cross_region_synchrony.png", dpi=300, bbox_inches='tight')
plt.show()
print("  ✅ Saved: O_cross_region_synchrony.png")

# =========================================================
# FINAL SUMMARY
# =========================================================
print("\n" + "="*60)
print("✅ ENHANCED EDA COMPLETE")
print("="*60)
charts = [
    "B_fire_density_normalised.png       — Area-normalised detection density",
    "C_month_year_heatmaps.png           — Year × Month fire count heatmaps",
    "D_doy_kde_curves.png                — Day-of-Year KDE fire density",
    "E_frp_violin_logbox.png             — FRP violin + log boxplot + Kruskal-Wallis",
    "F_confidence_deep_dive.png          — Confidence CDF, KDE, vs FRP scatter",
    "G_weather_correlation_matrices.png  — Weather variable correlation matrices",
    "H_fire_vs_weather_timeseries.png    — Fire counts vs weather time-series",
    "I_fire_vs_offseason_weather.png     — Fire/off-season weather T-test boxes",
    "J_fire_climatology_rolling.png      — 30-day rolling multi-year climatology",
    "K_fire_weather_scatter_matrix.png   — Fire count vs weather scatter + regression",
    "L_yoy_pct_change.png                — Year-on-year % change in detections",
    "M_engineered_features_dist.png      — Feature distributions (if features exist)",
    "M2_fire_nofire_separation.png       — Fire vs No-Fire distributions + Mann-Whitney",
    "N_spatial_fire_scatter.png          — Spatial lat/lon scatter coloured by FRP",
    "O_cross_region_synchrony.png        — Cross-region fire synchrony correlation",
]
for c in charts:
    print(f"   📊 {c}")
print(f"\n📁 All saved to: {OUTPUT_DIR}/")
print("\nNext step → Run: notebooks/03_spectral_index_engineering.ipynb")

In [ ]:
# ============================================================
# 02_EDA_SIMPLE.py  —  Beginner-Friendly EDA
# 8 clear, easy-to-understand charts about your wildfire data
# ============================================================

import os
import sys
import importlib.util
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# =========================================================
# 0️⃣  SETUP
# =========================================================
PROJECT_ROOT = r"C:\Users\vaast\OneDrive\Desktop\UAV_4_Case_Studies\wildfire-prediction-system"
os.chdir(PROJECT_ROOT)

config_path = os.path.join(PROJECT_ROOT, "src", "region_config.py")
spec = importlib.util.spec_from_file_location("region_config", config_path)
region_config = importlib.util.module_from_spec(spec)
sys.modules["region_config"] = region_config
spec.loader.exec_module(region_config)
REGIONS = region_config.REGIONS

OUTPUT_DIR = "outputs/comparative_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REGION_COLORS = {
    'laisong':    '#E63946',
    'jyotikuchi': '#457B9D',
    'corbett':    '#2A9D8F',
    'similipal':  '#E9C46A',
}
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

# =========================================================
# LOAD DATA
# =========================================================
regional_fires   = {}
regional_weather = {}

for region_name in REGIONS.keys():
    try:
        fires = pd.read_csv(
            f"data/case_studies/{region_name}/raw/firms/firms_{region_name}.csv")
        fires['acq_date'] = pd.to_datetime(fires['acq_date'])
        fires['year']     = fires['acq_date'].dt.year.astype(int)
        fires['month']    = fires['acq_date'].dt.month.astype(int)
        regional_fires[region_name] = fires

        weather = pd.read_csv(
            f"data/case_studies/{region_name}/raw/weather/daily_weather.csv")
        weather['date']  = pd.to_datetime(weather['date'])
        weather['month'] = weather['date'].dt.month.astype(int)
        weather['year']  = weather['date'].dt.year.astype(int)
        regional_weather[region_name] = weather

        print(f"  ✅ {REGIONS[region_name]['name']}: "
              f"{len(fires):,} fire detections | {len(weather):,} weather days")
    except FileNotFoundError as e:
        print(f"  ⚠️  Missing: {e}")

region_names  = list(regional_fires.keys())
region_labels = [REGIONS[rn]['name'] for rn in region_names]
colors        = [REGION_COLORS[rn] for rn in region_names]

print("\n🚀 Generating 8 charts...\n")


# =========================================================
# CHART 1 — How Many Fires Does Each Region Have?
# (Simple bar chart — total fire detections per region)
# =========================================================
totals = [len(regional_fires[rn]) for rn in region_names]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(region_labels, totals, color=colors, edgecolor='black', width=0.5)

# Put the number on top of each bar
for bar, val in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 30,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title("Chart 1: Total Fire Detections per Region (2018–2026)",
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Number of Fire Detections", fontsize=12)
ax.set_xlabel("Region", fontsize=12)
ax.tick_params(axis='x', rotation=10)

# Simple note explaining what this means
ax.text(0.01, 0.97,
        "💡 More detections = more fire activity recorded by satellite (NASA FIRMS)",
        transform=ax.transAxes, fontsize=9, color='dimgrey', va='top')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart1_total_detections.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Chart 1 saved: Total Detections per Region")


# =========================================================
# CHART 2 — Which Month Has the Most Fires?
# (Bar chart per region — monthly fire counts)
# =========================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, rn in enumerate(region_names):
    df  = regional_fires[rn]
    ax  = axes[idx]
    cfg = REGIONS[rn]

    monthly = df.groupby('month').size().reindex(range(1, 13), fill_value=0)

    # Colour fire-season months red, off-season blue
    fire_start = int(cfg['characteristics']['fire_season_start'])
    fire_end   = int(cfg['characteristics']['fire_season_end'])
    if fire_start <= fire_end:
        fire_months = set(range(fire_start, fire_end + 1))
    else:
        fire_months = set(range(fire_start, 13)) | set(range(1, fire_end + 1))

    bar_colors = ['orangered' if m in fire_months else 'steelblue'
                  for m in range(1, 13)]
    ax.bar(range(1, 13), monthly.values, color=bar_colors,
           edgecolor='black', linewidth=0.5)

    # Mark the peak month
    peak_m = int(monthly.idxmax())
    ax.bar(peak_m, monthly[peak_m], color='darkred', edgecolor='black', linewidth=0.5)
    ax.text(peak_m, monthly[peak_m] + monthly.max()*0.02,
            f"Peak\n{MONTH_LABELS[peak_m-1]}",
            ha='center', fontsize=8, color='darkred', fontweight='bold')

    ax.set_title(f"{REGIONS[rn]['name']}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Month", fontsize=9)
    ax.set_ylabel("Fire Detections", fontsize=9)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LABELS, fontsize=8)
    ax.grid(axis='y', alpha=0.3)

    # Legend
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor='orangered', label='Fire Season'),
                        Patch(facecolor='steelblue',  label='Off Season'),
                        Patch(facecolor='darkred',    label='Peak Month')],
              fontsize=8, loc='upper right')

fig.suptitle("Chart 2: Which Month Has the Most Fires? (Red = Fire Season)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart2_monthly_fire_pattern.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Chart 2 saved: Monthly Fire Patterns")


# =========================================================
# CHART 3 — How Has Fire Activity Changed Year by Year?
# (Line chart — annual totals for all regions)
# =========================================================
fig, ax = plt.subplots(figsize=(12, 6))

for rn in region_names:
    df     = regional_fires[rn]
    annual = df.groupby('year').size()
    ax.plot(annual.index, annual.values,
            marker='o', linewidth=2.5, markersize=7,
            color=REGION_COLORS[rn], label=REGIONS[rn]['name'])
    # Label the highest year
    peak_yr = annual.idxmax()
    ax.annotate(f"{peak_yr}\n({annual[peak_yr]:,})",
                xy=(peak_yr, annual[peak_yr]),
                xytext=(peak_yr + 0.1, annual[peak_yr] * 1.04),
                fontsize=7, color=REGION_COLORS[rn])

ax.set_title("Chart 3: Annual Fire Detections Over Time (2018–2026)",
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Total Fire Detections", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.text(0.01, 0.97,
        "💡 Peaks show bad fire years; valleys show calmer years",
        transform=ax.transAxes, fontsize=9, color='dimgrey', va='top')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart3_annual_trend.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Chart 3 saved: Annual Fire Trends")


# =========================================================
# CHART 4 — How Intense Are the Fires? (FRP)
# (Side-by-side bars: Average vs Maximum fire intensity)
# =========================================================
frp_data = pd.DataFrame([{
    'Region':   REGIONS[rn]['name'],
    'Average Intensity (Mean FRP)': round(regional_fires[rn]['frp'].mean(), 2),
    'Worst Fire (Max FRP)':         round(regional_fires[rn]['frp'].max(), 2),
} for rn in region_names])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Average FRP
axes[0].bar(frp_data['Region'], frp_data['Average Intensity (Mean FRP)'],
            color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title("Average Fire Intensity\n(Mean FRP in Megawatts)", fontweight='bold')
axes[0].set_ylabel("Fire Radiative Power (MW)")
axes[0].tick_params(axis='x', rotation=12)
for bar, val in zip(axes[0].patches, frp_data['Average Intensity (Mean FRP)']):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.1, f'{val} MW',
                 ha='center', fontsize=10, fontweight='bold')

# Right: Max FRP
axes[1].bar(frp_data['Region'], frp_data['Worst Fire (Max FRP)'],
            color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title("Most Intense Single Fire Ever\n(Max FRP in Megawatts)", fontweight='bold')
axes[1].set_ylabel("Fire Radiative Power (MW)")
axes[1].tick_params(axis='x', rotation=12)
for bar, val in zip(axes[1].patches, frp_data['Worst Fire (Max FRP)']):
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.5, f'{val} MW',
                 ha='center', fontsize=10, fontweight='bold')

fig.suptitle("Chart 4: How Intense Are the Fires?\n"
             "💡 Higher MW = Bigger, hotter fire",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart4_fire_intensity.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Chart 4 saved: Fire Intensity (FRP)")


# =========================================================
# CHART 5 — What Is the Weather Like During Fire Season?
# (4 simple bar charts: Temp, Humidity, Rain, Wind)
# =========================================================
def get_fire_months(rn):
    cfg = REGIONS[rn]['characteristics']
    s, e = int(cfg['fire_season_start']), int(cfg['fire_season_end'])
    return list(range(s, 13)) + list(range(1, e+1)) if s > e else list(range(s, e+1))

weather_summary = []
for rn, wdf in regional_weather.items():
    fm = get_fire_months(rn)
    fw = wdf[wdf['month'].isin(fm)]
    temp_col = next((c for c in wdf.columns if 'temperature_2m_mean' in c), None)
    rh_col   = next((c for c in wdf.columns if 'relative_humidity_2m_mean' in c), None)
    pr_col   = next((c for c in wdf.columns if 'precipitation_sum' in c), None)
    wd_col   = next((c for c in wdf.columns if 'wind_speed_10m_max' in c), None)
    weather_summary.append({
        'Region':      REGIONS[rn]['name'],
        'Temp (°C)':   round(fw[temp_col].mean(), 1) if temp_col else None,
        'Humidity (%)':round(fw[rh_col].mean(), 1)   if rh_col   else None,
        'Rain (mm)':   round(fw[pr_col].mean(), 2)   if pr_col   else None,
        'Wind (km/h)': round(fw[wd_col].mean(), 1)   if wd_col   else None,
    })

ws_df = pd.DataFrame(weather_summary)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
vars_  = ['Temp (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)']
clrs_  = ['tomato', 'cornflowerblue', 'mediumseagreen', 'mediumpurple']
notes_ = [
    "Higher temp → drier forest → more fire risk",
    "Lower humidity → drier air → more fire risk",
    "Less rain → drier ground → more fire risk",
    "Higher wind → fire spreads faster",
]

for i, (var, clr, note) in enumerate(zip(vars_, clrs_, notes_)):
    ax = axes[i]
    vals = ws_df[var].values
    bars = ax.bar(ws_df['Region'], vals, color=clr, edgecolor='black', alpha=0.85)
    ax.set_title(var, fontweight='bold', fontsize=12)
    ax.set_ylabel(var, fontsize=10)
    ax.tick_params(axis='x', rotation=12)
    ax.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+max(vals)*0.01,
                f'{v}', ha='center', fontsize=10, fontweight='bold')
    ax.text(0.01, 0.97, f"💡 {note}",
            transform=ax.transAxes, fontsize=8, color='dimgrey', va='top')

fig.suptitle("Chart 5: Average Weather During Fire Season — All Regions",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart5_weather_during_fire_season.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Chart 5 saved: Weather During Fire Season")


# =========================================================
# CHART 6 — How Reliable Are the Fire Detections?
# (Histogram of satellite confidence scores)
# =========================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, rn in enumerate(region_names):
    ax   = regional_fires[rn]['confidence'].dropna()
    axis = axes[idx]

    axis.hist(ax.values, bins=25, color=REGION_COLORS[rn],
              edgecolor='black', alpha=0.8)
    axis.axvline(80, color='red', linestyle='--', linewidth=2,
                 label='80% = High Confidence')
    axis.axvline(float(ax.mean()), color='navy', linestyle='-', linewidth=2,
                 label=f'Average = {ax.mean():.0f}%')

    pct_high = (ax > 80).mean() * 100
    axis.text(0.04, 0.92,
              f"{pct_high:.1f}% of detections\nabove 80% confidence",
              transform=axis.transAxes, fontsize=9,
              color='darkred', fontweight='bold', va='top')

    axis.set_title(f"{REGIONS[rn]['name']}", fontsize=11, fontweight='bold')
    axis.set_xlabel("Confidence Score (%)", fontsize=9)
    axis.set_ylabel("Number of Detections", fontsize=9)
    axis.legend(fontsize=8)
    axis.grid(axis='y', alpha=0.3)

fig.suptitle("Chart 6: How Confident Is the Satellite About Each Fire Detection?\n"
             "💡 Higher % = satellite is more sure it's a real fire (not a false alarm)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart6_detection_confidence.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Chart 6 saved: Detection Confidence")


# =========================================================
# CHART 7 — Does Temperature Affect Fire Activity?
# (Scatter plot: monthly avg temp vs monthly fire count)
# =========================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, rn in enumerate(region_names):
    ax   = axes[idx]
    fdf  = regional_fires[rn]
    wdf  = regional_weather[rn]

    temp_col = next((c for c in wdf.columns if 'temperature_2m_mean' in c), None)
    if temp_col is None:
        ax.set_visible(False)
        continue

    monthly_fires = fdf.groupby(['year','month']).size().reset_index(name='fires')
    monthly_temp  = wdf.groupby(['year','month'])[[temp_col]].mean().reset_index()
    merged = pd.merge(monthly_fires, monthly_temp, on=['year','month'])

    x = merged[temp_col].values
    y = merged['fires'].values

    ax.scatter(x, y, color=REGION_COLORS[rn], alpha=0.5, s=30, edgecolors='none')

    # Simple trend line
    if len(x) > 5:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, p(x_line), 'k--', linewidth=2, label='Trend')

    # Correlation
    corr = np.corrcoef(x, y)[0, 1]
    direction = "↑ more fires when hotter" if corr > 0 else "↓ fewer fires when hotter"
    ax.text(0.04, 0.94,
            f"Correlation: {corr:+.2f}\n{direction}",
            transform=ax.transAxes, fontsize=9,
            color='darkred', fontweight='bold', va='top')

    ax.set_title(f"{REGIONS[rn]['name']}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Monthly Average Temperature (°C)", fontsize=9)
    ax.set_ylabel("Fire Detections That Month", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle("Chart 7: Does Higher Temperature Mean More Fires?\n"
             "💡 Each dot = one month. Dashed line shows the trend.",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart7_temp_vs_fires.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Chart 7 saved: Temperature vs Fire Count")


# =========================================================
# CHART 8 — Quick Summary Dashboard
# (One-page overview of all 4 regions)
# =========================================================
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#f8f9fa')

# Title
fig.text(0.5, 0.96, "Chart 8: Wildfire Data — One-Page Summary Dashboard",
         ha='center', va='top', fontsize=16, fontweight='bold')
fig.text(0.5, 0.92, "4 regions | 2018–2026 | NASA FIRMS satellite detections",
         ha='center', va='top', fontsize=10, color='dimgrey')

# Build summary rows
rows = []
for rn in region_names:
    fdf = regional_fires[rn]
    wdf = regional_weather[rn]
    cfg = REGIONS[rn]
    temp_col = next((c for c in wdf.columns if 'temperature_2m_mean' in c), None)
    rh_col   = next((c for c in wdf.columns if 'relative_humidity_2m_mean' in c), None)
    fm = get_fire_months(rn)
    fw = wdf[wdf['month'].isin(fm)]
    peak_m = MONTH_LABELS[int(fdf.groupby('month').size().idxmax()) - 1]
    rows.append([
        cfg['name'],
        cfg['state'],
        f"{cfg['area_sq_km']:,} km²",
        f"{len(fdf):,}",
        peak_m,
        f"{fdf['frp'].mean():.1f} MW",
        f"{fdf['frp'].max():.0f} MW",
        f"{fdf['confidence'].mean():.0f}%",
        f"{fw[temp_col].mean():.1f} °C" if temp_col else "—",
        f"{fw[rh_col].mean():.0f}%"   if rh_col   else "—",
    ])

col_headers = ['Region', 'State', 'Area',
               'Total\nDetections', 'Peak\nMonth',
               'Avg\nIntensity', 'Max\nIntensity',
               'Avg\nConfidence',
               'Fire-Season\nTemp', 'Fire-Season\nHumidity']

ax_table = fig.add_axes([0.02, 0.38, 0.96, 0.50])
ax_table.axis('off')

tbl = ax_table.table(
    cellText=rows,
    colLabels=col_headers,
    cellLoc='center',
    loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 2.2)

# Style header row
for j in range(len(col_headers)):
    tbl[0, j].set_facecolor('#2E4057')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Style data rows with region colours
for i, rn in enumerate(region_names):
    hex_col = REGION_COLORS[rn]
    for j in range(len(col_headers)):
        tbl[i+1, j].set_facecolor(hex_col + '40')  # 25% opacity

# Add mini bar chart at bottom
ax_bar = fig.add_axes([0.05, 0.05, 0.40, 0.28])
totals_v = [len(regional_fires[rn]) for rn in region_names]
short_labels = [REGIONS[rn]['name'].split()[0] for rn in region_names]
ax_bar.bar(short_labels, totals_v, color=colors, edgecolor='black', alpha=0.85)
ax_bar.set_title("Total Detections", fontweight='bold', fontsize=10)
ax_bar.set_ylabel("Count")
for bar, val in zip(ax_bar.patches, totals_v):
    ax_bar.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+20, f'{val:,}',
                ha='center', fontsize=8, fontweight='bold')
ax_bar.grid(axis='y', alpha=0.3)

# Add mini line chart at bottom right
ax_line = fig.add_axes([0.55, 0.05, 0.42, 0.28])
for rn in region_names:
    annual = regional_fires[rn].groupby('year').size()
    ax_line.plot(annual.index, annual.values,
                 marker='o', ms=4, lw=2,
                 color=REGION_COLORS[rn],
                 label=REGIONS[rn]['name'].split()[0])
ax_line.set_title("Annual Trend", fontweight='bold', fontsize=10)
ax_line.set_xlabel("Year")
ax_line.set_ylabel("Detections")
ax_line.legend(fontsize=7)
ax_line.grid(alpha=0.3)

plt.savefig(f"{OUTPUT_DIR}/chart8_summary_dashboard.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Chart 8 saved: Summary Dashboard")


# =========================================================
# DONE
# =========================================================
print("\n" + "="*55)
print("✅ ALL 8 CHARTS COMPLETE")
print("="*55)
print(f"📁 Saved to: {OUTPUT_DIR}/")
print()
print("  Chart 1 — Total fire detections per region")
print("  Chart 2 — Which month has the most fires")
print("  Chart 3 — How fire activity changed year by year")
print("  Chart 4 — How intense are the fires (FRP)")
print("  Chart 5 — Weather during fire season")
print("  Chart 6 — How reliable are the detections (confidence)")
print("  Chart 7 — Does temperature affect fire count")
print("  Chart 8 — One-page summary dashboard")